In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ChannelAttention3D(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        """
        Channel attention module for 3D data
        
        Args:
            in_channels: Number of input channels
            reduction_ratio: Reduction ratio for the MLP
        """
        super(ChannelAttention3D, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.max_pool = nn.AdaptiveMaxPool3d(1)
        
        # Shared MLP for both pooled features
        self.mlp = nn.Sequential(
            nn.Conv3d(in_channels, in_channels // reduction_ratio, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv3d(in_channels // reduction_ratio, in_channels, kernel_size=1, bias=False)
        )
        
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Apply average pooling and max pooling
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        
        # Combine the features and apply sigmoid activation
        out = self.sigmoid(avg_out + max_out)
        
        return out


class SpatialAttention3D(nn.Module):
    def __init__(self, kernel_size=7):
        """
        Spatial attention module for 3D data
        
        Args:
            kernel_size: Size of the convolutional kernel
        """
        super(SpatialAttention3D, self).__init__()
        
        assert kernel_size in (3, 5, 7), "Kernel size must be 3, 5, or 7"
        padding = kernel_size // 2
        
        self.conv = nn.Conv3d(2, 1, kernel_size=(kernel_size, kernel_size, kernel_size), 
                             padding=(padding, padding, padding), bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Apply average pooling and max pooling along channel dimension
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        
        # Concatenate the features
        out = torch.cat([avg_out, max_out], dim=1)
        
        # Apply convolution and sigmoid activation
        out = self.conv(out)
        out = self.sigmoid(out)
        
        return out


class CBAM3D(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16, spatial_kernel_size=7):
        """
        Convolutional Block Attention Module (CBAM) for 3D data
        
        Args:
            in_channels: Number of input channels
            reduction_ratio: Reduction ratio for the channel attention MLP
            spatial_kernel_size: Kernel size for the spatial attention convolution
        """
        super(CBAM3D, self).__init__()
        
        self.channel_attention = ChannelAttention3D(in_channels, reduction_ratio)
        self.spatial_attention = SpatialAttention3D(spatial_kernel_size)
    
    def forward(self, x):
        # Store the input for the skip connection
        identity = x
        
        # Apply channel attention
        x = x * self.channel_attention(x)
        
        # Apply spatial attention
        x = x * self.spatial_attention(x)
        
        # Add skip connection
        x = x + identity
        
        return x


# Example usage
class ResBlock3D_with_CBAM(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResBlock3D_with_CBAM, self).__init__()
        
        # Main path
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)
        
        # CBAM attention module
        self.cbam = CBAM3D(out_channels)
        
        # Skip connection
        self.skip = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_channels)
            )
    
    def forward(self, x):
        # Store input for skip connection
        identity = x
        
        # Main path
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # Apply CBAM
        out = self.cbam(out)
        
        # Add skip connection
        out += self.skip(identity)
        
        # Final activation
        out = self.relu(out)
        
        return out


class CNN3D_with_CBAM(nn.Module):
    def __init__(self, in_channels=1, num_classes=10):
        super(CNN3D_with_CBAM, self).__init__()
        
        # Initial convolution
        self.conv1 = nn.Conv3d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        
        # Residual blocks with CBAM
        self.res1 = ResBlock3D_with_CBAM(64, 64)
        self.res2 = ResBlock3D_with_CBAM(64, 128, stride=2)
        self.res3 = ResBlock3D_with_CBAM(128, 256, stride=2)
        
        # Global average pooling and classifier
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(256, num_classes)
    
    def forward(self, x):
        # Initial convolution
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        # Residual blocks
        x = self.res1(x)
        x = self.res2(x)
        x = self.res3(x)
        
        # Global pooling and classification
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        
        return x
        


In [6]:
        
# Example tensor dimensions (batch_size, channels, depth, height, width)
# if __name__ == "__main__":
    # Example for 3D medical image or video data
in_channels = 12
# x = torch.randn(2, 12, 8, 8, 8)  # Batch of 2, 1 channel, 16 depth, 64x64 spatial dims

# model = CNN3D_with_CBAM(in_channels=in_channels, num_classes=2)
# output = model(x)

# # Calculate total parameters
# total_params = sum(p.numel() for p in model.parameters())

# print(f"Input shape: {x.shape}")
# print(f"Output shape: {output.shape}")
# print(f"Total parameters: {total_params:,}")

# Test a single residual block
res_block = ResBlock3D_with_CBAM(in_channels=in_channels, out_channels=128, stride=2)
test_input = torch.randn(2, in_channels, 8, 8, 8)
test_output = res_block(test_input)

print(f"\nResidual block test:")
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")




Residual block test:
Input shape: torch.Size([2, 12, 8, 8, 8])
Output shape: torch.Size([2, 128, 4, 4, 4])
